# Word2Vec (mean pooling) + Logistic Regression

**Task:** Abductive Commonsense Reasoning (ART). Given two observations O1, O2 and two
hypotheses H1, H2, pick the hypothesis that better explains what happened in between.

**Approach of this notebook:** represent every sentence as the *mean* of its pretrained
word2vec vectors and let a logistic regression decide between the two hypotheses.

In [1]:
%pip install -q datasets gensim nltk python-dotenv scikit-learn wandb

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from collections import Counter
from pathlib import Path

import gensim.downloader as api
import joblib
import nltk
import numpy as np
import wandb
from datasets import load_dataset
from dotenv import find_dotenv, load_dotenv
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# ~/nltk_data and are fetched at runtime. word_tokenize first splits the text into sentences
# with the Punkt model ("punkt_tab"), so without this line it raises a LookupError.
# i don't know why exactly yet - but the coding agent suggests it
nltk.download("punkt_tab", quiet=True)

True

In [2]:
# AI generated snipped reused from now on, ensuring i do not leak my keys
# Read the .env file in the project root. find_dotenv walks up from the notebook directory,
# load_dotenv keeps variables that are already set in the shell.
dotenv_path = find_dotenv(usecwd=True)
load_dotenv(dotenv_path)
print(f"environment loaded from: {dotenv_path or '<no .env found, using the defaults below>'}")

WANDB_BASE_URL = os.getenv("WANDB_BASE_URL", "https://api.wandb.ai")
WANDB_ENTITY = os.getenv("WANDB_ENTITY", "cmedi32-hochschule-luzern")
WANDB_PROJECT = os.getenv("WANDB_PROJECT", "nlp-project-01-abductive-reasoning")

os.environ["WANDB_BASE_URL"] = WANDB_BASE_URL
os.environ["WANDB_ENTITY"] = WANDB_ENTITY
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

wandb.login()

environment loaded from: /home/cedi/01_Projects/nlp-project-01-abductive-reasoning/.env


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: cmedi32 (cmedi32-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
config = {
    "dataset": "amarfurt/art-nlp-hs26",
    "tokenizer": "nltk.word_tokenize",  # label: the function used in the preprocessing cell below
    "lowercase": False,  # google news vectors are case sensitive -> keep the surface form
    "remove_punctuation": True, # as discussed in project discussions
    "remove_stopwords": False, # as discussed in project discussion
    "normalization": "none",  # no stemming, no lemmatization according to word2vec
    "truncation": "none",
    "embedding": "word2vec-google-news-300",  # gensim downloader model id: 300 dims, ~3M words, 1.7 GB
    "sentence_representation": "mean",
    "classifier": "logistic_regression",
}

run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name="w2v-mean-logreg",
    config=config,
)

In [4]:
dataset = load_dataset("amarfurt/art-nlp-hs26")

train = dataset["train"]
valid = dataset["validation"]
test = dataset["test"]

print(f"example observation printed")
print(f"train: {len(train)}, valid: {len(valid)}, test: {len(test)}")
print(train[0])

example observation printed
train: 16801, valid: 1000, test: 1532
{'observation_1': 'A baby panda was lost in the woods.', 'observation_2': 'He was reunited with his parents, who also lived at the zoo.', 'hypothesis_1': 'He was never lost.', 'hypothesis_2': 'All the people from the zoo looked for him.', 'label': 2}


### NLP pipeline

#### Preprocessing decisions

The target model is **pretrained word2vec (`word2vec-google-news-300`)**. Its vocabulary
is fixed and was built by somebody else, so we cannot adapt the model to our tokens — we
have to adapt our tokens to the model.

#### Filtering Tokens

raw text:  He was reunited with his parents, who also lived at the zoo.
tokenized: ['He', 'was', 'reunited', 'with', 'his', 'parents', ',', 'who', 'also', 'lived', 'at', 'the', 'zoo', '.']
filtered:  ['He', 'was', 'reunited', 'with', 'his', 'parents', 'who', 'also', 'lived', 'at', 'the', 'zoo']

In [5]:
def preprocess(text):
    # Turn a raw sentence into a list of tokens for word2vec. operation is elementary -> on the most atomic element to tokenize in this model its on word level
    tokens = nltk.word_tokenize(text)
    # Since we use punkt_tab tokenization - punctuation might also occur in the tokenized results, therefore we remove them by filtering via isalnum
    return [token for token in tokens if any(character.isalnum() for character in token)]

In [6]:
TEXT_FIELDS = ["observation_1", "observation_2", "hypothesis_1", "hypothesis_2"]

def preprocess_example(example):
    processed = {field: preprocess(example[field]) for field in TEXT_FIELDS}
    processed["target"] = example["label"] - 1 # we subtract 1 - so the classifier does not need to calculate that, the training data already indicates if its the first or the second hypothesis
    return processed


def preprocess_split(split):
    return split.map(preprocess_example)

In [19]:
train_tokens = preprocess_split(train)
valid_tokens = preprocess_split(valid)
test_tokens = preprocess_split(test)

A quick look at one example, before and after preprocessing. This is the check that the
decisions above actually took effect: punctuation is gone, casing and stopwords are
untouched, word forms are unchanged, nothing is cut off.

In [8]:
for field in TEXT_FIELDS:
    print(f"{field}:")
    print(f"  train:    {train[0][field]}")
    print(f"  train_tokens: {train_tokens[0][field]}")

observation_1:
  train:    A baby panda was lost in the woods.
  train_tokens: ['A', 'baby', 'panda', 'was', 'lost', 'in', 'the', 'woods']
observation_2:
  train:    He was reunited with his parents, who also lived at the zoo.
  train_tokens: ['He', 'was', 'reunited', 'with', 'his', 'parents', 'who', 'also', 'lived', 'at', 'the', 'zoo']
hypothesis_1:
  train:    He was never lost.
  train_tokens: ['He', 'was', 'never', 'lost']
hypothesis_2:
  train:    All the people from the zoo looked for him.
  train_tokens: ['All', 'the', 'people', 'from', 'the', 'zoo', 'looked', 'for', 'him']


In [9]:
wv = api.load("word2vec-google-news-300")
print(f"vocabulary: {len(wv.index_to_key)} words, {wv.vector_size} dimensions")

vocabulary: 3000000 words, 300 dimensions


In [10]:
oov_counts = Counter()  # oov token -> how often it occurred
embed_stats = Counter()  # "tokens", "oov", "empty_sentences"

def embed(tokens):
    vectors = []
    for token in tokens:
        embed_stats["tokens"] += 1
        if token in wv:
            vectors.append(wv[token])
        elif token.lower() in wv:
            vectors.append(wv[token.lower()])
        else:
            embed_stats["oov"] += 1
            oov_counts[token] += 1
    if not vectors:
        embed_stats["empty_sentences"] += 1
        return np.zeros(wv.vector_size)
    return np.mean(vectors, axis=0)

In [11]:
def embed_example(example):
    vectors = [embed(example[field]) for field in TEXT_FIELDS]
    return np.mean(vectors, axis=0) # → 1200d and predict label 1 vs 2 directly. simpler could also have one feature per hypotheses

In [12]:
X_train = np.stack([embed_example(example) for example in train_tokens])
y_train = np.array(train_tokens["target"])
X_valid = np.stack([embed_example(example) for example in valid_tokens])
y_valid = np.array(valid_tokens["target"])

# Decision Remaining: Whether the averaged vectors are L2-normalized before the logistic regression.

classifier = LogisticRegression(max_iter=1000) # simple regression, puts out weight tensor and bias term
classifier.fit(X_train, y_train)

train_accuracy = classifier.score(X_train, y_train)
valid_accuracy = classifier.score(X_valid, y_valid)

print(f"train accuracy: {train_accuracy:.2%}")
print(f"valid accuracy: {valid_accuracy:.2%}")

train accuracy: 54.89%
valid accuracy: 52.70%


In [13]:
WEIGHTS_DIR = Path("../../../models/project_1/w2v_mean_logreg/weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(classifier, WEIGHTS_DIR / "logreg.joblib") # output classifiers weights / ai generated, not sure if i need it at the moment

['../../../models/project_1/w2v_mean_logreg/weights/logreg.joblib']

In [14]:
def forward(split):
    vector = embed_example(split[0])
    probabilities = classifier.predict_proba(vector.reshape(1, -1))[0]
    return probabilities

probabilities = forward(valid_tokens)
print(probabilities)

[0.50851972 0.49148028]


In [15]:
wandb.log({
    "oov/tokens": embed_stats["tokens"],
    "oov/occurrences": embed_stats["oov"],
    "oov/rate": embed_stats["oov"] / embed_stats["tokens"],
    "oov/unique_types": len(oov_counts),
    "oov/empty_sentences": embed_stats["empty_sentences"],
    "accuracy/train": train_accuracy,
    "accuracy/valid": valid_accuracy,
})

In [16]:
X_test = np.stack([embed_example(example) for example in test_tokens])
y_test = np.array(test_tokens["target"])

test_accuracy = classifier.score(X_test, y_test)

majority_accuracy = np.bincount(y_test).max() / len(y_test)
oov_rate = embed_stats["oov"] / embed_stats["tokens"]

In [17]:
predictions = classifier.predict(X_test)

In [18]:
wandb.summary.update({
    "accuracy/train": train_accuracy,
    "accuracy/valid": valid_accuracy,
    "accuracy/test": test_accuracy,
    "accuracy/majority": majority_accuracy,
    "oov/rate": oov_rate,
})

wandb.finish()

accuracy/train,▁
accuracy/valid,▁
oov/empty_sentences,▁
oov/occurrences,▁
oov/rate,▁
oov/tokens,▁
oov/unique_types,▁
accuracy/majority,0.50979
accuracy/test,0.51567
accuracy/train,0.5489
accuracy/valid,0.527
